# (II) Detection and Picking
This notebook demonstrates the use of EQTransformer for performing the earthquake signal detection and seismic phase (P & S) picking on continuous data. Once you have your seismic data - preferentially in mseed format and in individual subfolders for each station- you can perform the detection/picking using the following options:


### Option (I) on preprocessed (hdf5) files:

This option is recommended for smaller time periods (a few days to a month). This allows you to test the perfomance and explore the effects of different parameters while the provided hdf5 file makes it easy to access the waveforms.

For this option you first need to convert your MiniSeed files for each station into a single hdf5 file and a csv file containting the list of traces in the hdf5 file.

You can convert MiniSeed files to a hdf5 file using the following command:

In [1]:
import os
from EQTransformer.utils.hdf5_maker import preprocessor

json_basepath = os.path.join(os.getcwd(),"json/station_list.json")

preprocessor(preproc_dir="preproc",
             mseed_dir='downloads_mseeds', 
             stations_json=json_basepath, 
             overlap=0.3, 
             n_processor=2)

Using TensorFlow backend.


 *** " /home/sysop/eqt/examples/downloads_mseeds_processed_hdfs " directory already exists!


 * --> Do you want to creat a new empty folder? Type (Yes or y)  y


============ Station B921 has 2 chunks of data.
============ Station CA06 has 2 chunks of data.
  * B921 (1) .. 20190901 --> 20190902 .. 2 components .. sampling rate: 100.0
  * CA06 (1) .. 20190901 --> 20190902 .. 1 components .. sampling rate: 100.0
  * CA06 (2) .. 20190902 --> 20190903 .. 2 components .. sampling rate: 100.0
  * B921 (2) .. 20190902 --> 20190903 .. 3 components .. sampling rate: 100.0
 Station CA06 had 2 chuncks of data
4112 slices were written, 4114.0 were expected.
Number of 1-components: 1. Number of 2-components: 1. Number of 3-components: 0.
Original samplieng rate: 100.0.
============ Station SV08 has 2 chunks of data.
  * SV08 (1) .. 20190901 --> 20190902 .. 3 components .. sampling rate: 100.0
 Station B921 had 2 chuncks of data
4112 slices were written, 4114.0 were expected.
Number of 1-components: 0. Number of 2-components: 1. Number of 3-components: 1.
Original samplieng rate: 100.0.
  * SV08 (2) .. 20190902 --> 20190903 .. 1 components .. sampling rate: 

This will generate one "station_name.hdf5" and one "station_name.csv" file for each of your stations and put them into a directory named "mseed_dir+_hdfs". Then you need to pass the name of the directory containing your hdf5 & CSV files and a model. You can use relatively low threshold values for the detection and picking since EQTransformer is very robust to false positives. Enabling uncertaintiy estimation, outputing probabilities, or plotting all the detected events will slow down the process.

In [2]:
from EQTransformer.core.predictor import predictor
predictor(input_dir='downloads_mseeds_processed_hdfs',   
         input_model='../ModelsAndSampleData/EqT_original_model.h5',
         output_dir='detections1',
         estimate_uncertainty=False, 
         output_probabilities=False,
         number_of_sampling=5,
         loss_weights=[0.02, 0.40, 0.58],          
         detection_threshold=0.3,                
         P_threshold=0.3,
         S_threshold=0.3, 
         number_of_plots=10,
         plot_mode='time',
         batch_size=500,
         number_of_cpus=4,
         keepPS=False,
         spLimit=60) 

Running EqTransformer  0.1.59
 *** Loading the model ...



2025-09-17 03:54:53.469241: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-09-17 03:54:53.482026: E tensorflow/stream_executor/cuda/cuda_driver.cc:318] failed call to cuInit: UNKNOWN ERROR (303)
2025-09-17 03:54:53.482091: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (2db13e998484): /proc/driver/nvidia/version does not exist
2025-09-17 03:54:53.616971: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 1999990000 Hz
2025-09-17 03:54:53.624996: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x86f75b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-09-17 03:54:53.625037: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version



*** Loading is complete!
 *** /home/sysop/eqt/examples/detections1 already exists!


 --> Type (Yes or y) to create a new empty directory! otherwise it will overwrite!    y


######### There are files for 3 stations in downloads_mseeds_processed_hdfs directory. #########
========= Started working on B921, 1 out of 3 ...
100%|█████████████████████████████████████████████████████████████████| 9/9 [01:44<00:00,  9.91s/it]

 *** Finished the prediction in: 0 hours and 1 minutes and 46.5 seconds.
 *** Detected: 5309 events.
 *** Wrote the results into --> " /home/sysop/eqt/examples/detections1/B921_outputs "
========= Started working on CA06, 2 out of 3 ...

100%|█████████████████████████████████████████████████████████████████| 9/9 [01:03<00:00,  7.18s/it]

 *** Finished the prediction in: 0 hours and 1 minutes and 6.17 seconds.
 *** Detected: 5196 events.
 *** Wrote the results into --> " /home/sysop/eqt/examples/detections1/CA06_outputs "
========= Started working on SV08, 3 out of 3 ...


  0%|                                                                         | 0/9 [00:00<?, ?it/s]

 22%|██████████████▍                                                  

If you are using local MiniSeed files you can generate a station_list.json by supplying an absolute path to a directory containing Miniseed files and a station location dictionary using the stationListFromMseed function like the following:

In [3]:
from EQTransformer.utils.plot import plot_data_chart
plot_data_chart('time_tracks.pkl', time_interval=10)

FileNotFoundError: [Errno 2] No such file or directory: 'time_tracks.pkl'

In [6]:
from EQTransformer.utils.hdf5_maker import stationListFromMseed

mseed_directory = '/home/sysop/eqt/examples/downloads_mseeds'
station_locations = {"B921": [35.5865, -117.4622, 694.5], "CA06": [35.59962, -117.49268, 796.4], "SV08": [35.5761, -117.4187, 604.0]}
stationListFromMseed(mseed_directory, station_locations)

### Option (II) directly on downloaded MiniSeed files:

You can perform the detection/picking directly on .mseed files. 
This save both prerpcessing time and the extra space needed for hdf5 file. However, it can be more memory intensive. So it is recommended when mseed fils are one month long or shorter.
This option also does not allow you to estimate the uncertainties, write the prediction probabilities, or use the advantages of having hdf5 files which makes it easy to access the raw event waveforms based on detection results.   

In [2]:
from EQTransformer.core.mseed_predictor import mseed_predictor
mseed_predictor(input_dir='downloads_mseeds',   
         input_model='../ModelsAndSampleData/EqT_original_model.h5',
         stations_json=json_basepath,
         output_dir='detections2',
         loss_weights=[0.02, 0.40, 0.58],          
         detection_threshold=0.7,                
         P_threshold=0.3,
         S_threshold=0.3, 
         number_of_plots=10,
         plot_mode='time_frequency',
         normalization_mode='std',
         batch_size=500,
         overlap=0.9,
         gpuid=None,
         gpu_limit=None) 

09-17 03:45 [INFO] [EQTransformer] Running EqTransformer  0.1.59
09-17 03:45 [INFO] [EQTransformer] *** Loading the model ...


09-17 03:45 [WARNING] [tensorflow] From /home/sysop/miniconda3/envs/eqt/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:4070: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.

2025-09-17 03:45:31.173728: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-09-17 03:45:31.178276: E tensorflow/stream_executor/cuda/cuda_driver.cc:318] failed call to cuInit: UNKNOWN ERROR (303)
2025-09-17 03:45:31.178332: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (2db13e998484): /proc/driver/nvidia/version does not exist
2025-09-17 03:45:31.253599: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 1999990000 Hz
2025-09-17 03:45:31.268579: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x87d79a0 initialized for pl

09-17 03:45 [WARNING] [tensorflow] From /home/sysop/miniconda3/envs/eqt/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:422: The name tf.global_variables is deprecated. Please use tf.compat.v1.global_variables instead.

09-17 03:46 [INFO] [EQTransformer] *** Loading is complete!
09-17 03:46 [INFO] [EQTransformer] There are files for 3 stations in downloads_mseeds directory.
09-17 03:46 [INFO] [EQTransformer] Started working on B921, 1 out of 3 ...
09-17 03:46 [INFO] [EQTransformer] 20190901T000000Z__20190902T000000Z.mseed
09-17 03:48 [DEBUG] [matplotlib.pyplot] Loaded backend agg version unknown.
09-17 03:48 [DEBUG] [matplotlib.font_manager] findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0.
09-17 03:48 [DEBUG] [matplotlib.font_manager] findfont: score(FontEntry(fname='/home/sysop/miniconda3/envs/eqt/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/STIXNonUniBol.ttf', name='STIXNonUnicode', style='normal', variant

TypeError: Dimensions of C (41, 151) are incompatible with X (151) and/or Y (41); see help(pcolormesh)

Prediction outputs for each station will be written in your output directory (i.e. 'detections').

'X_report.txt' contains processing info on input parameters used for the detection/picking and final 
results such as running time, the total number of detected events (these are unique events and duplicated ones have been already removed). 

'X_prediction_results.csv' contains detection/picking results in the figures folder you can find the plots for the number of events that you specified in the above comment.